# 00 - понять LIBERO как LeRobotDataset

Цель notebook: пока просто скачать и проверить совместимость форматов

In [12]:
# import os
# os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

from tqdm import tqdm_notebook
from datasets import load_dataset
from pathlib import Path
from PIL import Image
import shutil
from huggingface_hub import snapshot_download
from lerobot.datasets import LeRobotDataset
import torch

ROOT_DIR = Path.cwd().resolve().parent
DATA_DIR = ROOT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True, parents=True)


В задании просят работать с `libero_90` и `libero_goal` (`task_ids`=0,1,2). \
Значит скачаем их ниже

In [2]:
DATA_DIR = ROOT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True, parents=True)

In [3]:
# snapshot_download(
#     repo_id="nvidia/LIBERO_LeRobot_v3",
#     repo_type="dataset",
#     local_dir=str(DATA_DIR),
#     allow_patterns=[
#         "libero_90/**",
#         "libero_goal/**",
#     ],
#     max_workers=16,
# )

Fetching 50 files: 100%|██████████| 50/50 [27:52<00:00, 33.45s/it] 


'/home/msi/projects/TrainerVLA/data'

In [ ]:
# удалю кеш, чтобы не занимал место
# shutil.rmtree(DATA_DIR / ".cache")

In [3]:
LIBERO_90_DIR = DATA_DIR / "libero_90"
LIBERO_GOAL_DIR = DATA_DIR / "libero_goal"

In [4]:
ds90 = LeRobotDataset(
    repo_id="nvidia/LIBERO_LeRobot_v3",
    root=LIBERO_90_DIR,
)

ds_goal = LeRobotDataset(
    repo_id="nvidia/LIBERO_LeRobot_v3",
    root=LIBERO_GOAL_DIR,
)

In [27]:
sample = ds90[0]
for k in sample.keys():
    print(k)

observation.images.image
observation.images.wrist_image
observation.state
observation.states.ee_state
observation.states.joint_state
observation.states.gripper_state
action
timestamp
frame_index
episode_index
index
task_index
task


In [31]:
img_tensor = sample["observation.images.image"]
img_tensor.min(), img_tensor.max(), img_tensor.dtype, img_tensor.shape 

(tensor(0.), tensor(1.), torch.float32, torch.Size([3, 256, 256]))

In [6]:
import pandas as pd
from libero.libero import benchmark



# 1. То, что использует LeRobot env при --env.task=libero_goal --env.task_ids=[...]
suite = benchmark.get_benchmark_dict()["libero_goal"]()

print("LIBERO / LeRobot env task_ids:")
for env_task_id in range(10):
    task = suite.get_task(env_task_id)
    print(f"  env task_id={env_task_id}: {task.language}")

# 2. То, что лежит в локальном HF/NVIDIA parquet dataset
tasks = pd.read_parquet("../data/libero_goal/meta/tasks.parquet")

print("\nHF/NVIDIA data/libero_goal task_index:")
for text, row in tasks.sort_values("task_index").iterrows():
    print(f"  dataset task_index={int(row.task_index)}: {text}")

# 3. Соответствие между ними по тексту задачи
env_id_by_text = {
    suite.get_task(i).language: i
    for i in range(10)
}

print("\nMapping: HF/NVIDIA task_index -> LIBERO env task_id")
for text, row in tasks.sort_values("task_index").iterrows():
    env_id = env_id_by_text.get(str(text))
    print(f"  dataset task_index={int(row.task_index)} -> env task_id={env_id}: {text}")


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
LIBERO / LeRobot env task_ids:
  env task_id=0: open the middle drawer of the cabinet
  env task_id=1: put the bowl on the stove
  env task_id=2: put the wine bottle on top of the cabinet
  env task_id=3: open the top drawer and put the bowl inside
  env task_id=4: put the bowl on top of the cabinet
  env task_id=5: push the plate to the front of the stove
  env task_id=6: put the cream cheese in the bowl
  env task_id=7: turn on the stove
  env task_id=8: put the bowl on the plate
  env task_id=9: put the wine bottle on the rack

HF/NVIDIA data/libero_goal task_index:
  dataset task_index=0: put the bowl on the plate
  dataset task_index=1: put the wine bottle on the rack
  dataset task_index=2: open the top drawer and put the bowl inside
  dataset task_index=3: put the cream cheese in the bowl
  dataset task_index=4: put the wine bottle on top of the cabinet
  dataset task_index=5: push the plate to the front of the stove
  data

Узнаем какие эпизоды соответсвуют каким задачам 

In [15]:
from pathlib import Path
import pandas as pd
from libero.libero.benchmark import libero_task_map

K = 25

tasks = pd.read_parquet("../data/libero_goal/meta/tasks.parquet")
tasks.index.name = "task"

df = pd.concat(
    pd.read_parquet(p, columns=["episode_index", "task_index"])
    for p in sorted(Path("../data/libero_goal/data").glob("**/*.parquet"))
)

episodes = (
    df.drop_duplicates(["episode_index", "task_index"])
      .sort_values("episode_index")
)

for env_task_id in [0, 1, 2]:
    task_text = libero_task_map["libero_goal"][env_task_id].replace("_", " ")
    dataset_task_index = int(tasks.loc[task_text, "task_index"])

    first_eps = (
        episodes[episodes["task_index"] == dataset_task_index]["episode_index"]
        .head(K)
        .astype(int)
        .tolist()
    )

    print(f"\nLIBERO env task_id={env_task_id}")
    print(f"text: {task_text}")
    print(f"dataset task_index={dataset_task_index}")
    print(f"--dataset.episodes='{first_eps}'")
    print(f"--env.task_ids='[{env_task_id}]'")


LIBERO env task_id=0
text: open the middle drawer of the cabinet
dataset task_index=9
--dataset.episodes='[20, 26, 31, 42, 58, 64, 84, 90, 94, 111, 117, 118, 137, 140, 146, 162, 168, 173, 182, 187, 198, 206, 220, 232, 252]'
--env.task_ids='[0]'

LIBERO env task_id=1
text: put the bowl on the stove
dataset task_index=7
--dataset.episodes='[13, 15, 16, 22, 36, 45, 66, 76, 116, 121, 145, 151, 165, 166, 171, 178, 179, 186, 201, 219, 225, 233, 237, 239, 250]'
--env.task_ids='[1]'

LIBERO env task_id=2
text: put the wine bottle on top of the cabinet
dataset task_index=4
--dataset.episodes='[6, 10, 17, 18, 25, 38, 40, 44, 48, 51, 53, 57, 75, 89, 91, 96, 97, 100, 101, 103, 133, 136, 149, 154, 164]'
--env.task_ids='[2]'
